# M3 Agentic AI - Email assistant workflow

## 1. Introduction

### 1.1 Lab overview
Andrew showed a calendar assistant that uses multiple tools to carry out complex tasks. Here you work with a similar **email assistant agent**: it can send emails, search for emails from a specific sender, delete emails and more. You give it natural-language instructions, like *"check unread emails from my boss"* or *"delete the Happy Hour email"*, and see how it selects the right tools and completes the task.

### 🎯 1.2 Learning outcome
Connect an LLM to tools with aisuite, give natural-language instructions, and observe how the agent selects, executes and validates multi-step tasks such as searching, sending and deleting emails.

> **Before you start:** copy `.env.example` (in the `ToolUseDemo` folder) to `.env` and add your `OPENAI_API_KEY`.

## 2. Initialize environment and client

In [ ]:
# ================================
# Imports
# ================================

# --- Third-party ---
from dotenv import load_dotenv
import aisuite as ai
import json

# --- Local / project ---
import utils
import display_functions
import email_tools



# ================================
# Environment & Client
# ================================
load_dotenv()          # Load environment variables from .env
client = ai.Client()   # Initialize AISuite client

## 3. Simulated email service

### 3.1 Components
A simulated email backend, your personal sandbox inbox preloaded with messages. It started automatically in the background when you imported `utils` / `email_tools` (see `email_service.py`; API docs at http://127.0.0.1:8765/docs).

| Layer | Purpose |
|---|---|
| FastAPI | Exposes REST endpoints |
| SQLite + SQLAlchemy | Stores and queries emails locally (`emails.db`) |
| Pydantic | Ensures inputs and outputs are valid |
| aisuite tools | Bridge between the LLM and the service |

### 3.2 Endpoints
- `POST /send` → send a new email
- `GET /emails` → list all emails
- `GET /emails/unread` → show only unread emails
- `GET /emails/{id}` → fetch a specific email by ID
- `GET /emails/search?q=...` → search emails by keyword
- `GET /emails/filter` → filter by recipient or date range
- `PATCH /emails/{id}/read` → mark an email as read
- `PATCH /emails/{id}/unread` → mark an email as unread
- `DELETE /emails/{id}` → delete an email by ID
- `GET /reset_database` → reset emails to initial state (for testing)

💡 **Key idea:** these endpoints are the "inbox controls". Next you'll expose them as Python functions (tools) the LLM can call.

### 3.3 Endpoint test helpers
The `utils.test_*` functions are quick wrappers around the endpoints. 👉 Uncomment the lines you'd like to try, run the cell, and check the results.

In [ ]:
# uncomment the line 'utils.test_*' you want to try
new_email_id = utils.test_send_email()
_ = utils.test_get_email(new_email_id['id'])
#_ = utils.test_list_emails()
#_ = utils.test_filter_emails(recipient="test@example.com")
#_ = utils.test_search_emails("lunch")
#_ = utils.test_unread_emails()
#_ = utils.test_mark_read(new_email_id['id'])
#_ = utils.test_mark_unread(new_email_id['id'])
#_ = utils.test_delete_email(new_email_id['id'])
#_ = utils.reset_database()

## 4. Tool layer for the email agent

### 4.1 Why tools?
Each tool wraps a REST route, turning raw API calls into actions the agent can perform. Tools are the agent's *actuators*: you give an instruction and the model chooses which tools to call, and in what order.

### 4.2 Design hints
- Keep tool docstrings short, imperative, and specific to the action.
- Return consistent, compact JSON so the model can chain results.
- Prefer one clear responsibility per tool (single route, single effect).

### 4.3 Available tools
| Tool | Action |
|---|---|
| `list_all_emails()` | Fetch all emails, newest first |
| `list_unread_emails()` | Retrieve only unread emails |
| `search_emails(query)` | Search by keyword in subject, body, or sender |
| `filter_emails(...)` | Filter by recipient and/or date range |
| `get_email(email_id)` | Fetch a specific email by ID |
| `mark_email_as_read(id)` | Mark an email as read |
| `mark_email_as_unread(id)` | Mark an email as unread |
| `send_email(...)` | Send a new (simulated) email |
| `delete_email(id)` | Delete an email by ID |
| `search_unread_from_sender(addr)` | Unread emails from a given sender (e.g. boss@email.com) |

👉 Uncomment the ones you'd like to run, execute the cell, and review the outputs.

In [ ]:
# Test sending a new email and fetch it by ID
new_email = email_tools.send_email("test@example.com", "Lunch plans", "Shall we meet at noon?")
content_ = email_tools.get_email(new_email['id'])

# Uncomment the ones you want to try:
#content_ = email_tools.list_all_emails()
#content_ = email_tools.list_unread_emails()
#content_ = email_tools.search_emails("lunch")
#content_ = email_tools.filter_emails(recipient="test@example.com")
#content_ = email_tools.mark_email_as_read(new_email['id'])
#content_ = email_tools.mark_email_as_unread(new_email['id'])
#content_ = email_tools.search_unread_from_sender("test@example.com")
#content_ = email_tools.delete_email(new_email['id'])

utils.print_html(content=json.dumps(content_, indent=2), title="Testing the email_tools")

## 5. Preparing the agent prompt
`build_prompt()` wraps your request in a preamble so the LLM knows it is an email assistant, may use the tools, and acts **without asking for confirmation** (no human-in-the-loop).

In [ ]:
def build_prompt(request_: str) -> str:
    return f"""
- You are an AI assistant specialized in managing emails.
- You can perform various actions such as listing, searching, filtering, and manipulating emails.
- Use the provided tools to interact with the email system.
- Never ask the user for confirmation before performing an action.
- If needed, my email address is "you@email.com" so you can use it to send emails or perform actions related to my account.

{request_.strip()}
"""

See how it wraps a raw user prompt:

In [ ]:
example_prompt = build_prompt("Delete the Happy Hour email")
utils.print_html(content=example_prompt, title="Example example_prompt")

### 5.3 Resetting the email service
Reset the inbox to its initial state before handing control to the agent.

In [ ]:
utils.reset_database()

## 6. LLM + Email tools

### 6.1 Scenario
*"Check for unread emails from boss@email.com, mark them as read, and send a polite follow-up."*

### 6.2 What happens
1. The agent interprets your instruction.
2. It selects the right tools (`search_unread_from_sender` → `mark_email_as_read` → `send_email`).
3. It executes each action automatically.

aisuite handles schema exposure, argument binding, execution, and passing results between steps.

### 6.3 Run it
Look for a clear tool-call trace (which tools, which arguments) and a concise final summary.

In [ ]:
# Try your own requests
prompt_ = build_prompt("Check for unread emails from boss@email.com, mark them as read, and send a polite follow-up.")

response = client.chat.completions.create(
    model="openai:gpt-4.1",
    messages=[{"role": "user", "content": (
        prompt_
    )}],
    tools=[ # list of tools that the LLM can access
        email_tools.search_unread_from_sender,
        email_tools.list_unread_emails,
        email_tools.search_emails,
        email_tools.get_email,
        email_tools.mark_email_as_read,
        email_tools.send_email
    ],
    max_turns=5
)

display_functions.pretty_print_chat_completion(response)

*(Extra)* Verify in the inbox: no unread boss emails left, and a follow-up was sent.

In [ ]:
_ = utils.test_unread_emails()
_ = utils.test_filter_emails(recipient="boss@email.com")

### 6.4. Missing tool: `delete_email`
What happens if the tool you need isn't available? Try *"Delete alice@work.com's email"*: `delete_email` is not registered, so the LLM responds but can't complete the action.

🔍 **Key insight:** the tools you make available directly determine what the agent can do.

In [ ]:
# Try with a request that may call an unavailable tool
prompt_ = build_prompt("Delete alice@work.com email")

response = client.chat.completions.create(
    model="openai:o4-mini",
    messages=[{"role": "user", "content": (
        prompt_
    )}],
    tools=[ # list of tools that the LLM can access
        email_tools.search_unread_from_sender,
        email_tools.list_unread_emails,
        email_tools.search_emails,
        email_tools.get_email,
        email_tools.mark_email_as_read,
        email_tools.send_email
    ],
    max_turns=5
)

display_functions.pretty_print_chat_completion(response)

*(Extra)* Verify: Alice's email is still in the inbox.

In [ ]:
_ = utils.test_search_emails("alice@work.com")

#### 6.4.1 Try again with `delete_email` enabled
Add `delete_email` to the tool list and re-run. **Tip:** watch the sequence of calls: after finding the message, the agent should select `delete_email`.

In [ ]:
prompt_ = build_prompt("Delete alice@work.com email")

response = client.chat.completions.create(
    model="openai:o4-mini",
    messages=[{"role": "user", "content": (
        prompt_
    )}],
    tools=[
        email_tools.search_unread_from_sender,
        email_tools.list_unread_emails,
        email_tools.search_emails,
        email_tools.get_email,
        email_tools.mark_email_as_read,
        email_tools.send_email,
        email_tools.delete_email
    ],
    max_turns=5
)

display_functions.pretty_print_chat_completion(response)

*(Extra)* Verify: Alice's email is gone.

In [ ]:
_ = utils.test_search_emails("alice@work.com")

### 6.5. Targeted action: Delete the "Happy Hour" email
The inbox is preloaded with this message:
```json
{
  "id": 1,
  "sender": "eric@work.com",
  "recipient": "you@email.com",
  "subject": "Happy Hour",
  "body": "We're planning drinks this Friday!",
  "timestamp": "2025-06-13T04:48:59.096908",
  "read": false
}
```
Watch how the agent searches for the message and deletes it.

In [ ]:
prompt_ = build_prompt("Delete the happy hour email")

response = client.chat.completions.create(
    model="openai:o4-mini",
    messages=[{"role": "user", "content": (
        prompt_
    )}],
    tools=[
        email_tools.search_unread_from_sender,
        email_tools.list_unread_emails,
        email_tools.search_emails,
        email_tools.get_email,
        email_tools.mark_email_as_read,
        email_tools.send_email,
        email_tools.delete_email
    ],
    max_turns=5
)

display_functions.pretty_print_chat_completion(response)

*(Extra)* Verify: the Happy Hour email is gone.

In [ ]:
_ = utils.test_search_emails("happy hour")

## 7. Final Takeaways
- Tool calling lets LLMs go beyond text generation: they call functions and complete multi-step tasks.
- The set of available tools determines what the agent can and cannot do (without `delete_email`, it cannot remove messages).
- Clear docstrings and consistent behavior help the LLM select the right tool for each step.
- aisuite manages the interaction layer: exposing Python functions as tools, binding parameters, and returning results.
- Observing the full workflow (prompt → tool calls → outputs → final response) is key to understanding and improving agents.